# Webinar 4 — Recopilación de datos
**TripleTen · Sprint 12**

---
## Antes de arrancar

> *Una empresa de salud entrena un modelo para predecir si un paciente va a reingresar al hospital en los próximos 30 días. Exactitud: 97%. Lo lanzan. Tres meses después, en producción: 61%.*

**¿Qué crees que salió mal? Escribe en el chat.**

---

## Agenda

| # | Bloque |
|---|--------|
| 1 | Fuentes de datos y exploración | 
| 2 | Data leakage | 
| 3 | Pipeline + Cross validation |
| 4 | GridSearch |
| 5 | Feature importances | 
| 6 | Modelo final | 

**Al terminar vas a poder:**
- Detectar data leakage antes de entrenar
- Construir un pipeline completo de preprocesamiento + modelo
- Evaluar con cross validation y afinar con GridSearch
- Identificar qué features aportan y cuáles son ruido
- Entregar un modelo final listo para producción

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.metrics import classification_report

np.random.seed(42)

---
# PARTE 1 — Fuentes de datos y exploración

Dataset de churn de telecomunicaciones — fuente abierta en GitHub.
El objetivo es predecir si un cliente va a cancelar el servicio (`Churn = 1`).

In [5]:
csv_telecom = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
df  = pd.read_csv(csv_telecom)

print(f'Shape:  {df.shape}')
print(f'Nulos:  {df.isnull().sum().sum()}')
print(f'Churn:  {df["Churn"].value_counts().to_dict()}')
df.head()

Shape:  (7043, 21)
Nulos:  0
Churn:  {'No': 5174, 'Yes': 1869}


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# Codificamos las dos columnas categóricas y el target
le = LabelEncoder()
for col in ['International plan', 'Voice mail plan', 'Churn']:
    df[col] = le.fit_transform(df[col])

df = df.drop(columns=['State', 'Area code'])

features = df.drop(columns=['Churn'])
target   = df['Churn']

print(f'Features: {features.shape[1]} columnas')
print(f'Target:   {target.value_counts().to_dict()}')

---
# PARTE 2 — Data Leakage

Volvamos al caso del inicio. El modelo vio información que en producción no existiría.

**¿Cuál columna tiene la fuga? Escribe en el chat antes de ejecutar la siguiente celda.**

In [ ]:
ejemplo = pd.DataFrame({
    'paciente_id':         [1,            2,    3,            4,    5,    6],
    'edad':                [67,           45,   72,           38,   55,   61],
    'dias_internacion':    [8,            2,    12,           1,    4,    9],
    'diagnostico_cronico': [1,            0,    1,            0,    1,    1],
    'fecha_reingreso':     ['2024-03-10', None, '2024-04-02', None, None, '2024-05-18'],
    'reingreso_30d':       [1,            0,    1,            0,    0,    1]
})
print(ejemplo.to_string(index=False))

---
# PARTE 3 — Pipeline

Un pipeline encadena el preprocesamiento y el modelo en un solo objeto.
Ventaja clave: cuando llamas `.predict()`, el preprocesamiento se aplica automáticamente — no hay riesgo de olvidar el `transform` en producción.

In [ ]:
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   RandomForestClassifier(random_state=42))
])

print(pipeline)

### Cross validation sobre el pipeline

Evaluamos el pipeline completo — no las features crudas, sino el flujo entero incluyendo el preprocesamiento.

In [ ]:
scores = cross_val_score(pipeline, features, target, cv=5, scoring='accuracy')

print('Exactitud por fold:')
for i, s in enumerate(scores):
    print(f'  Fold {i+1}: {"█" * int(s*40)} {s:.4f}')
print(f'\n  Promedio: {scores.mean():.4f}')
print(f'  Std:      {scores.std():.4f}')

---
# PARTE 4 — GridSearch

Con el baseline validado, buscamos los mejores hiperparámetros.
GridSearch prueba todas las combinaciones con cross validation interno — no necesitamos correr `cross_val_score` de nuevo después.

In [ ]:
# Los parámetros del pipeline se nombran como 'paso__parametro'
param_grid = {
    'model__n_estimators':  [100, 200],
    'model__max_depth':     [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(features, target)

print(f'\nMejores parámetros: {grid_search.best_params_}')
print(f'Mejor exactitud CV: {grid_search.best_score_:.4f}')

---
# PARTE 5 — Feature Importances

Con el modelo afinado, vemos qué features aportan realmente.
Las importancias indican cuánto contribuye cada columna a las decisiones del Random Forest.

In [ ]:
# Extraemos el modelo del mejor pipeline
best_model = grid_search.best_estimator_.named_steps['model']

importances = pd.Series(
    best_model.feature_importances_,
    index=features.columns
).sort_values(ascending=False)

print('Top 10 features:')
print(importances.head(10).to_string())

In [ ]:
top10 = importances.head(10)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#4C9BE8' if v > importances.mean() else '#94A3B8' for v in top10]
ax.barh(top10.index[::-1], top10.values[::-1], color=colors[::-1], edgecolor='none')
ax.axvline(importances.mean(), color='#F4845F', linestyle='--', linewidth=1.5, label=f'Media: {importances.mean():.3f}')
ax.set_xlabel('Importancia')
ax.set_title('Feature Importances — Top 10', fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'\nFeatures por debajo de la media ({importances.mean():.3f}) → candidatas a eliminar:')
print(importances[importances < importances.mean()].index.tolist())

---
# PARTE 6 — Modelo final

Entrenamos con los mejores parámetros sobre todo el dataset.
Sin dividir en folds — aprovechamos todos los datos disponibles.

In [ ]:
# Seleccionamos solo las features que superan la media de importancia
features_seleccionadas = importances[importances >= importances.mean()].index.tolist()
print(f'Features seleccionadas ({len(features_seleccionadas)}): {features_seleccionadas}')

In [ ]:
# Pipeline final con las features seleccionadas
X_final = features[features_seleccionadas]

modelo_final = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   RandomForestClassifier(**{
        k.replace('model__', ''): v
        for k, v in grid_search.best_params_.items()
    }, random_state=42))
])

modelo_final.fit(X_final, target)
print('Modelo final entrenado ✅')
print(f'Features: {len(features_seleccionadas)}')
print(f'Muestras de entrenamiento: {len(target)}')

In [ ]:
# Evaluación rápida sobre train (referencia) + reporte completo
y_pred = modelo_final.predict(X_final)
print(classification_report(target, y_pred, target_names=['No churn', 'Churn']))

---
# Resumen

| Paso | Qué hace | Para qué sirve |
|------|----------|----------------|
| **Data leakage** | Detectar columnas del futuro | Evitar modelos que hacen trampa |
| **Pipeline** | Encadenar preprocesamiento + modelo | Reproducible y seguro en producción |
| **Cross validation** | Evaluar el baseline con K folds | Saber si el enfoque es razonable |
| **GridSearch** | Buscar mejores hiperparámetros | Afinar sin perder generalización |
| **Feature importances** | Ver qué columnas aportan | Reducir ruido y simplificar el modelo |
| **Modelo final** | Entrenar con todo el dataset | Aprovechar al máximo los datos |

---
### Volvamos al caso del inicio

> El modelo pasó de 97% a 61% en producción.

La columna `fecha_reingreso` estaba en el dataset de entrenamiento pero no existe al momento de predecir. El pipeline no lo detecta solo — es responsabilidad del data scientist revisar cada feature antes de entrenar.

**Regla:** antes de meter una columna al modelo, preguntá — *¿esta información estaría disponible en el momento real de la predicción?*